In [1]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

# The Halting Problem

This notebook proves the undecidability of the *halting problem*.
In this proof, programs are represented as *strings*.  Hence we need a way to turn a string into a function that we can call.  In Python, this is done by the built-in function `exec`.  F# is a compiled language and has no such function.  However, the notebook kernel that executes our F# code is able to compile and execute any F# code that is given as a string.  Therefore, we start by implementing a function `exec` that sends a string to the notebook kernel.

The function `exec` takes one argument:
- `code` is a string containing F# code, e.g. the definition of a function.

If `code` can be compiled without errors, then `code` is executed and all definitions contained in `code` become available in the following cells of this notebook.  Otherwise, the compiler errors are displayed and the message *"This is not a valid F# definition."* is printed.  The function does not return a meaningful result.

**Implementation:** You do not need to understand the details of this function in order to follow the rest of the notebook.
1. First, we ask the F# kernel to compile `code` and to report all errors.  This is done by sending the command `RequestDiagnostics(code)` to the kernel.  This command does not execute `code`.  The kernel returns a list of *events*.  Events of type `DiagnosticsProduced` contain the *diagnostics*, i.e. the error messages and warnings produced by the compiler.  We collect all diagnostics that are errors.
2. If there are no errors, we send the command `SubmitCode(code)` to the kernel.  This command compiles and executes `code`.

   There is a subtle point here:  The command is not executed immediately.  Instead, it is executed after the execution of the current cell has finished.  The reason is that the F# kernel would forget all definitions made in `code` once the current cell finishes, if `code` was executed while the current cell is still running.
   This is achieved as follows: `Task.Run` starts the submission in the background and `ExecutionContext.SuppressFlow()` ensures that the kernel does not regard this submission as part of the current cell.  Therefore, the kernel queues the submission and runs it as soon as the current cell is finished.

In [2]:
open System.Threading
open System.Threading.Tasks
open Microsoft.DotNet.Interactive
open Microsoft.DotNet.Interactive.Commands
open Microsoft.DotNet.Interactive.Events

let exec (code: string) : unit =
    let kernel = Kernel.Root.FindKernelByName "fsharp"
    let result = kernel.SendAsync(RequestDiagnostics(code)).Result
    let errors =
        [ for event in result.Events do
            match event with
            | :? DiagnosticsProduced as d ->
                for diagnostic in d.Diagnostics do
                    if string diagnostic.Severity = "Error" then
                        diagnostic.Message
            | _ -> () ]
    if errors.IsEmpty then
        use _ = ExecutionContext.SuppressFlow()
        Task.Run(fun () -> kernel.SendAsync(SubmitCode(code)) :> Task) |> ignore
    else
        printfn "This is not a valid F# definition."

We continue with a number of definitions.

A string $t$ is a *test function with name $f$*
iff $t$ has the form
```
"""
let f (x: string) =
    body
"""
```
and, furthermore, the string $t$ can be compiled as an *F#* function, that is the evaluation of
```
exec t
```
does not yield an error.  Note that in F#, compiling a function includes *type checking*.  Hence a string that is syntactically correct but contains a type error is not a test function.  Furthermore, by definition the argument `x` of a test function is a string.
The set of all test functions is denoted as $T\!F$.  If $t \in T\!F$ and $t$ has the name $f$, then
this is written as $\mathtt{name}(t) = f$.

We provide some examples next.  We use *triple quoted strings*, i.e. strings that start and end with `"""`.  These strings can extend over several lines.

The string `t1` is a *test function* with the name `simple`.

In [3]:
let t1 = """
let simple (x: string) =
    0
"""

In [4]:
printfn "%s" t1


let simple (x: string) =
    0


By executing this string, we define the function `simple`.

In [5]:
exec t1

In [6]:
simple "1"

0

The string `t2` is a *test function* with the name `loop`.  The type of `loop` is `string -> unit`, as the function never returns a value.

In [7]:
let t2 = """
let loop (x: string) =
    let mutable n = 0
    while true do
        n <- n + 1
"""

In [8]:
exec t2

Obviously, the next cell will not terminate.

In [ ]:
loop "1"

The string `t3` is not a *test function* since F# does not have the operator `++`.

In [9]:
let t3 = """
let hugo (x: string) =
    x++
"""

As `t3` is not a test function, the next cell will yield a *syntax error*.

In [10]:
exec t3


/Users/karl/Library/Mobile Documents/com~apple~CloudDocs/Box/Kurse/Logic/F-Sharp/Chapter-2/stdin.fsx (4,1)-(4,1) parse error Unexpected syntax or possible incorrect indentation: this token is offside of context started at position (3:5). Try indenting this further.
To continue using non-conforming indentation, pass the '--strict-indentation-' flag to the compiler, or set the language version to F# 7.

/Users/karl/Library/Mobile Documents/com~apple~CloudDocs/Box/Kurse/Logic/F-Sharp/Chapter-2/stdin.fsx (3,6)-(3,8) parse error Unexpected token '++' or incomplete expression



This is not a valid F# definition.

The string `t4` is not a test function either.  It is syntactically correct, but it contains a *type error*: the argument `x` is a string, but the operator `-` can not be applied to a string.

In [11]:
let t4 = """
let otto (x: string) =
    x - 1
"""

In [12]:
exec t4


/Users/karl/Library/Mobile Documents/com~apple~CloudDocs/Box/Kurse/Logic/F-Sharp/Chapter-2/stdin.fsx (3,9)-(3,10) typecheck error The type 'int' does not match the type 'string'

/Users/karl/Library/Mobile Documents/com~apple~CloudDocs/Box/Kurse/Logic/F-Sharp/Chapter-2/stdin.fsx (3,7)-(3,8) typecheck error The type 'int' does not match the type 'string'



This is not a valid F# definition.

**Notation:**
If $n$ is the name of an F# function that takes $k$ arguments $a_1$, $\cdots$, $a_k$,
then we write
$$n(a_1, \cdots, a_k) \leadsto r$$
iff the evaluation of the expression $n\;a_1\;\cdots\;a_k$ yields the result $r$.

If we are not
concerned with the result $r$ but only want to state that the evaluation *terminates* eventually,
then we will write
$$n(a_1, \cdots, a_k) \,\downarrow$$
and read this notation as *evaluation of $n(a_1, \cdots, a_k)$ terminates*.

If the evaluation of the expression $n(a_1, \cdots, a_k)$ does **not** terminate, this is
written as
$$n(a_1, \cdots, a_k) \,\uparrow.$$
This notation is read as *evaluation of $n(a_1, \cdots, a_k)$ diverges*.

The *halting problem* for F# functions is the question whether there is an
F# function
```
let stops (t: string) (a: string) : int =
    ...
```
that takes as input a string $t$ and a string $a$ and that satisfies the following specification:
- $t \not\in T\!F \quad\Leftrightarrow\quad \mathtt{stops}(t, a) \leadsto 2$.

  If the first argument of `stops` is not a test function, then
  `stops t a = 2`.
- $t \in T\!F \,\wedge\, \mathtt{name}(t) = n \,\wedge\, n(a)\downarrow \quad\Leftrightarrow\quad \mathtt{stops}(t, a) \leadsto 1$.

  If the first argument of `stops` is a test function with name $n$ and, furthermore,
  the evaluation of $n(a)$ terminates, then `stops t a = 1`.

- $t \in T\!F \,\wedge\, \mathtt{name}(t) = n \,\wedge\, n(a)\uparrow \quad\Leftrightarrow\quad \mathtt{stops}(t, a) \leadsto 0$.

  If the first argument of `stops` is a test function with name $n$ but the evaluation of $n(a)$
  diverges, then `stops t a = 0`.

The string `turing` is a test function with name `alan`.
It makes use of the function `stops`.  The function `alan` takes one argument:
- `x` is a string.

It computes `stops x x`.  If the result is `1`, then `alan` enters an infinite loop.  Otherwise, `alan` returns the result of `stops x x`.

In [13]:
let turing = """
let alan (x: string) : int =
    let result = stops x x
    if result = 1 then
        while true do
            printfn "... looping ..."
    result
"""

As `turing` is a test function, there are only two possibilities for the call `stops turing turing`.
1. Case:  `stops turing turing = 1`.
   Provided the function `stops` solves the *halting problem*, then the call `alan turing` should terminate.
2. Case:  `stops turing turing = 0`.
   According to the specification of `stops`, then the call `alan turing` should diverge.

Let us check both cases.  First, we pretend that `stops turing turing = 1`.

In [14]:
let stops (t: string) (a: string) : int =
    1

Now we can execute the string `turing`.  In F#, a function can only use functions that have already been defined.  Hence, we had to define `stops` before executing `turing`.

In [15]:
exec turing

We will see that contrary to the specification of `stops`, the call `alan turing` loops.

**Warning:** The next cell does not terminate.  It prints the line `... looping ...` forever.

In [ ]:
alan turing

Let us assume that `stops turing turing = 0` next.

In [16]:
let stops (t: string) (a: string) : int =
    0

In F#, names are bound *statically*:  The function `alan` defined above uses the version of `stops` that was visible when `alan` was defined.  This version of `stops` always returns `1`.  The new definition of `stops` creates a new function that *shadows* the old one, but `alan` still refers to the old function.  Therefore, we have to execute the string `turing` again, so that `alan` uses the new version of `stops`.

In [17]:
exec turing

According to the specification of the function `stops`, the call `alan turing` should loop.  However, it returns `0` instead.

In [18]:
alan turing

0

We can only conclude that there is no way that the function `stops` can satisfy its specification.
Hence the *halting problem* is unsolvable.